In [33]:
library(tidyverse)
train <- read.csv("https://raw.githubusercontent.com/blacktreeM/econ/refs/heads/main/train_weekly_sales.csv")
test <- read.csv("https://raw.githubusercontent.com/blacktreeM/econ/refs/heads/main/test_weekly_sales.csv")

In [34]:
nrow(train) # 476333

[1] 476333

In [35]:
colnames(train) # 'Date''ID''Weekly_Sales'
colnames(test) # 'Date''ID'

[1] "Date"         "ID"           "Weekly_Sales"

[1] "Date" "ID"

In [36]:
# Define holiday weeks
holidays <- as.Date(c("2010-02-12","2011-02-11","2012-02-10","2013-02-08",
                      "2010-09-10","2011-09-09","2012-09-07","2013-09-06",
                      "2010-11-26","2011-11-25","2012-11-23","2013-11-29",
                      "2010-12-31","2011-12-30","2012-12-28","2013-12-27"))

train$Date <- as.Date(train$Date)
test$Date  <- as.Date(test$Date)

# Holiday flag
train$Holiday <- as.integer(train$Date %in% holidays)
test$Holiday  <- as.integer(test$Date  %in% holidays)

In [37]:
# Numeric time index for trend
all_dates  <- sort(unique(c(train$Date, test$Date)))
date_index <- setNames(seq_along(all_dates), all_dates)
train$t    <- date_index[as.character(train$Date)]
test$t     <- date_index[as.character(test$Date)]

In [38]:
# Week number
train$week <- as.integer(format(train$Date, "%U"))
test$week  <- as.integer(format(test$Date,  "%U"))

In [39]:
# Time columns (train only — not needed in test for numeric ones)
train$year    <- as.integer(format(train$Date, "%Y"))
train$month   <- as.integer(format(train$Date, "%m"))
train$days    <- as.integer(format(train$Date, "%d"))

In [40]:
# Categorical columns — factor with shared levels
train$day     <- factor(weekdays(train$Date))
train$quarter <- factor(quarters(train$Date))
test$day      <- factor(weekdays(test$Date), levels = levels(train$day))
test$quarter  <- factor(quarters(test$Date), levels = levels(train$quarter))
test$year     <- as.integer(format(test$Date, "%Y"))
test$month    <- as.integer(format(test$Date, "%m"))

In [43]:
train <- train %>%
  group_by(ID) %>%
  arrange(Date) %>%
  mutate(
    lag_1 = lag(Weekly_Sales, 1),
    lag_2 = lag(Weekly_Sales, 2),
    roll_4 = slider::slide_dbl(Weekly_Sales, mean, .before = 3, .complete = FALSE)
  ) %>%
  ungroup()

In [44]:
predictors <- c(
  "week",
  "month",
  "year",
  "Holiday",
  "lag_1",
  "lag_2",
  "roll_4"
)

In [45]:
# Loop over each ID and predict
ids <- unique(test$ID)
preds <- map_dfr(ids, function(id) {
  tr <- train[train$ID == id, ]
  te <- test[test$ID == id, ]

  # Drop factor columns with fewer than 2 levels for this ID's subset
  cat_cols  <- c("day", "quarter")
  valid_cats <- cat_cols[sapply(cat_cols, function(col) nlevels(droplevels(tr[[col]])) >= 2)]
  predictors <- c(predictors, valid_cats)
  fmla       <- as.formula(paste("Weekly_Sales ~", paste(predictors, collapse = " + ")))

  fit <- if (nrow(tr) >= 8) {
    tryCatch(lm(fmla, data = tr), error = function(e) NULL)
  } else NULL

  yhat <- if (!is.null(fit)) predict(fit, newdata = te) else mean(tr$Weekly_Sales, na.rm = TRUE)
  yhat <- ifelse(is.na(yhat), 0, yhat)

  data.frame(
    Id           = paste(id, format(te$Date, "%Y-%m-%d"), sep = "_"),
    Weekly_Sales = yhat
  )
})


ERROR: [1m[33mError[39m in `map()`:[22m
[1m[22m[36mℹ[39m In index: 1.
[1mCaused by error:[22m
[33m![39m object 'lag_1' not found


In [42]:
write.csv(preds, "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Classes Spring 2026/Econometrics/submissions/Walmart Recruiting - Store Sales Forecasting/x.csv", row.names = FALSE)

In [48]:
library(dplyr)
library(purrr)
library(lubridate)
library(slider)
library(tidyr)

# -----------------------------
# 1. SAFE ID CREATION (FIXED BUG SOURCE)
# -----------------------------
train$Store <- as.character(train$Store)
train$Dept  <- as.character(train$Dept)
test$Store  <- as.character(test$Store)
test$Dept   <- as.character(test$Dept)

train <- train %>% filter(!is.na(Store), !is.na(Dept))
test  <- test  %>% filter(!is.na(Store), !is.na(Dept))

train$ID <- paste0(train$Store, "_", train$Dept)
test$ID  <- paste0(test$Store, "_", test$Dept)

# -----------------------------
# 2. FEATURE ENGINEERING (TRAIN ONLY LAGS)
# -----------------------------
train <- train %>%
  group_by(ID) %>%
  arrange(Date) %>%
  mutate(
    week  = week(Date),
    month = month(Date),
    year  = year(Date),
    Holiday = as.numeric(IsHoliday),

    lag_1 = lag(Weekly_Sales, 1),
    lag_2 = lag(Weekly_Sales, 2),
    lag_4 = lag(Weekly_Sales, 4),

    roll_4 = slide_dbl(Weekly_Sales, mean, .before = 3, .complete = FALSE)
  ) %>%
  ungroup()

test <- test %>%
  mutate(
    week  = week(Date),
    month = month(Date),
    year  = year(Date),
    Holiday = as.numeric(IsHoliday),

    lag_1 = 0,
    lag_2 = 0,
    lag_4 = 0,
    roll_4 = 0
  )

# -----------------------------
# 3. CLEAN DATA
# -----------------------------
train <- train %>% mutate(across(where(is.numeric), ~replace_na(., 0)))
test  <- test  %>% mutate(across(where(is.numeric), ~replace_na(., 0)))

# -----------------------------
# 4. MODELING
# -----------------------------
ids <- unique(test$ID)

preds <- map_dfr(ids, function(id) {

  tr <- train[train$ID == id, ]
  te <- test[test$ID == id, ]

  predictors <- c(
    "week",
    "month",
    "year",
    "Holiday",
    "lag_1",
    "lag_2",
    "lag_4",
    "roll_4"
  )

  fmla <- as.formula(
    paste("Weekly_Sales ~", paste(predictors, collapse = " + "))
  )

  fit <- if (nrow(tr) >= 8) {
    tryCatch(lm(fmla, data = tr), error = function(e) NULL)
  } else NULL

  yhat <- if (!is.null(fit)) {
    predict(fit, newdata = te)
  } else {
    rep(mean(tr$Weekly_Sales, na.rm = TRUE), nrow(te))
  }

  # -----------------------------
  # SAFE CLEANING (IMPORTANT)
  # -----------------------------
  yhat <- as.numeric(yhat)
  yhat[is.na(yhat)] <- mean(tr$Weekly_Sales, na.rm = TRUE)
  yhat[yhat < 0] <- 0

  # -----------------------------
  # FIXED ID (NO NA, NO DUPLICATES)
  # -----------------------------
  data.frame(
    Id = paste0(id, "_", format(te$Date, "%Y-%m-%d")),
    Weekly_Sales = yhat
  )
})

# -----------------------------
# 5. FINAL CLEANUP
# -----------------------------
preds <- preds %>%
  filter(!is.na(Id)) %>%
  distinct(Id, .keep_all = TRUE)

preds$Weekly_Sales <- as.numeric(preds$Weekly_Sales)
preds$Weekly_Sales[is.na(preds$Weekly_Sales)] <- 0


Warning message:
"package 'slider' was built under R version 4.5.3"
Warning message:
"Unknown or uninitialised column: `Store`."


ERROR: [1m[33mError[39m in `$<-`:[22m
[1m[22m[33m![39m Assigned data `as.character(train$Store)` must be compatible with existing data.
[31m✖[39m Existing data has 476333 rows.
[31m✖[39m Assigned data has 0 rows.
[36mℹ[39m Only vectors of size 1 are recycled.
[1mCaused by error in `vectbl_recycle_rhs_rows()`:[22m
[33m![39m Can't recycle input of size 0 to size 476333.


In [47]:
write.csv(preds, "C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Classes Spring 2026/Econometrics/submissions/Walmart Recruiting - Store Sales Forecasting/25.csv", row.names = FALSE)